In [0]:
#reading the silver tables
from pyspark.sql import functions as F

transactions = spark.read.table("projects.ecommerce_project.silver_transactons")
adjustments = spark.read.table("projects.ecommerce_project.silver_adjustments")
completed_transactions = transactions.filter(~F.col("iscancelled"))


In [0]:
#First gold table to find the monthly revenue by country
monthly_df = transactions.withColumn("year_month", F.date_format("invoice_date","yyyy-MM"))\
                        .groupBy("year_month","country")\
                        .agg(F.sum(F.when(~F.col("iscancelled"), F.col("total_price")).otherwise(0)).alias("Net_revenue"),
                             F.countDistinct(~F.col("iscancelled"), F.col("invoice_no")).alias("completed_orders"),
                             F.countDistinct(F.col("iscancelled"),F.col("invoice_no")).alias("cancelled_orders"),
                             F.countDistinct("invoice_No").alias("total_orders"))\
                        .withColumn("cancellation_rate", F.round(F.col("cancelled_orders")/F.col("total_orders"),2))\
                        .withColumn("avg_order_value", F.round(F.col("Net_revenue")/F.col("total_orders"),2))\
                        .orderBy("year_month","country")                     

display(monthly_df)

In [0]:
#Second Gold Table to  find the top products by country
from pyspark.sql.window import Window
top_products = completed_transactions.groupBy("stock_code","description")\
                                    .agg(
                                        F.sum("total_price").alias("total_revenue"),
                                        F.sum("quantity").alias("total_quantity_sold"),
                                        F.countDistinct("invoice_no").alias("num_orders"),
                                        F.countDistinct("customer_id").alias("num_unique_customers")
                                    )\
                                    .withColumn("revenue_rank", F.row_number().over(Window.orderBy(F.desc("total_revenue")))) \
                                    .orderBy(F.desc("total_revenue"))

display(top_products)


In [0]:
max_date = completed_transactions.agg(F.max("invoice_date")).collect()[0][0]
reference_date = F.lit(max_date) + F.expr("INTERVAL 1 DAY")

rfm = completed_transactions \
    .filter(F.col("customer_id").isNotNull()) \
    .groupBy("customer_id") \
    .agg(
        F.datediff(reference_date, F.max("invoice_date")).alias("recency_days"),
        F.countDistinct("invoice_no").alias("frequency"),
        F.sum("total_price").alias("monetary")
    )

# Score each dimension into quartiles (1=worst, 4=best)
from pyspark.sql.window import Window

rfm = rfm \
    .withColumn("r_score", 5 - F.ntile(4).over(Window.orderBy("recency_days"))) \
    .withColumn("f_score", F.ntile(4).over(Window.orderBy("frequency"))) \
    .withColumn("m_score", F.ntile(4).over(Window.orderBy("monetary"))) \
    .withColumn("rfm_score", F.col("r_score") + F.col("f_score") + F.col("m_score")) \
    .withColumn("segment",
        F.when(F.col("rfm_score") >= 10, "Champions")
         .when(F.col("rfm_score") >= 7, "Loyal Customers")
         .when(F.col("rfm_score") >= 5, "At Risk")
         .otherwise("Lost")
    )

display(rfm.orderBy(F.desc("rfm_score")).limit(20))

In [0]:
cancellation_summary = transactions \
    .withColumn("year_month", F.date_format("invoice_date", "yyyy-MM")) \
    .groupBy("year_month", "country", "stock_code", "description") \
    .agg(
        F.countDistinct(F.when(F.col("iscancelled"), F.col("invoice_no"))).alias("cancelled_order_count"),
        F.sum(F.when(F.col("iscancelled"), F.abs(F.col("total_price"))).otherwise(0)).alias("cancelled_value"),
        F.countDistinct("invoice_no").alias("total_order_count")
    ) \
    .filter(F.col("cancelled_order_count") > 0) \
    .withColumn("cancellation_rate", F.round(F.col("cancelled_order_count") / F.col("total_order_count"), 4)) \
    .orderBy(F.desc("cancelled_value"))

display(cancellation_summary.limit(20))

In [0]:
monthly_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("projects.ecommerce_project.gold_monthly_revenue_by_country")

top_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("projects.ecommerce_project.gold_top_products")

rfm.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("projects.ecommerce_project.gold_customer_rfm")

cancellation_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("projects.ecommerce_project.gold_cancellation_summary")